In [1]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from word2vec import Word2Vec
from utils import SkipGramPairsDataset

In [2]:
BATCH_SIZE = 256
EMBEDDING_DIM = 300
NEGATIVE_SAMPLES = 5
EPOCHS = 5
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
base_dir = Path.cwd()
pares_path = os.path.join(base_dir, "..","datasets", "pares_skipgram_direito.csv")
print("Carregando os pares de palavras...")
print("Pares path:", pares_path)

df_pares = pd.read_csv(pares_path)
df_pares = df_pares[(df_pares["centro"] != 0) & (df_pares["contexto"] != 0)].reset_index(drop=True)

VOCAB_SIZE = int(max(df_pares["centro"].max(), df_pares["contexto"].max())) + 1

print("Device:", DEVICE)
print("VOCAB_SIZE:", VOCAB_SIZE)
print("Quantidade de pares:", len(df_pares))

Carregando os pares de palavras...
Pares path: /home/brunofernandes/git/t2_introducao_pln_word2vec_similarity/src/../datasets/pares_skipgram_direito.csv
Device: cuda
VOCAB_SIZE: 2001
Quantidade de pares: 115231318


In [4]:
dataset = SkipGramPairsDataset(df_pares)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

if len(dataset) == 0:
    raise ValueError("Dataset de pares está vazio após o filtro de UNK.")

In [5]:
def sample_negative_ids(batch_size, num_negatives, vocab_size, device, positive_ids=None):
    negatives = torch.randint(
        low=1,
        high=vocab_size,
        size=(batch_size, num_negatives),
        device=device,
    )

    if positive_ids is not None:
        positive_ids = positive_ids.unsqueeze(1)
        mask = negatives.eq(positive_ids)
        while mask.any():
            negatives[mask] = torch.randint(
                low=1,
                high=vocab_size,
                size=(mask.sum().item(),),
                device=device,
            )
            mask = negatives.eq(positive_ids)

    return negatives

In [6]:
word2vec_model = Word2Vec(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM).to(DEVICE)
optimizer = torch.optim.Adam(word2vec_model.parameters(), lr=LEARNING_RATE)

In [7]:
loss_history = []

word2vec_model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0

    for center_ids, pos_context_ids in loader:
        center_ids = center_ids.to(DEVICE)
        pos_context_ids = pos_context_ids.to(DEVICE)

        neg_context_ids = sample_negative_ids(
            batch_size=center_ids.size(0),
            num_negatives=NEGATIVE_SAMPLES,
            vocab_size=VOCAB_SIZE,
            device=DEVICE,
            positive_ids=pos_context_ids,
        )

        optimizer.zero_grad()
        loss = word2vec_model(center_ids, pos_context_ids, neg_context_ids)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_loss = total_loss / len(loader)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss média: {epoch_loss:.4f}")

Epoch 1/5 - loss média: 1.4820
Epoch 2/5 - loss média: 1.4742
Epoch 3/5 - loss média: 1.4757
Epoch 4/5 - loss média: 1.4775
Epoch 5/5 - loss média: 1.4784


In [8]:
output_dir = base_dir.parent / "datasets"
output_dir.mkdir(parents=True, exist_ok=True)

torch.save(word2vec_model.state_dict(), output_dir / "word2vec_model.pt")
torch.save(word2vec_model.get_input_embeddings().detach().cpu(), output_dir / "word_embeddings_input.pt")
torch.save(word2vec_model.get_output_embeddings().detach().cpu(), output_dir / "word_embeddings_output.pt")

pd.DataFrame({"epoch_loss": loss_history}).to_csv(output_dir / "word2vec_loss_history.csv", index=False)

print("Modelo, embeddings e histórico de loss salvos.")

Modelo, embeddings e histórico de loss salvos.
